In [ ]:
# Gans ETL Pipeline

This project collects data from multiple external sources,
transforms it using pandas,
and stores it in a MySQL database.

Sources:
- Wikipedia (web scraping)
- OpenWeather API
- AeroDataBox API

In [ ]:
import pandas as pd
import requests
from bs4 import BeautifulSoup

In [ ]:
#Wikipedia Berlin
url = "https://en.wikipedia.org/wiki/Berlin"
headers = {'User-Agent': 'Chrome/134.0.0.0'}

response = requests.get(url, headers=headers)

berlin = BeautifulSoup(response.content, 'html.parser')


In [ ]:
# Find just the right-hand information card
infobox = berlin.find("table", class_="infobox")

# Print only the first 500 characters of the infobox HTML
print(infobox.prettify()[:500])

In [ ]:
ll_='infobox-data'
berlin.find_all(class_='infobox-data')[4]

# Grab the table rows
infobox_rows = berlin.find("table", class_="infobox").find_all("tr")

# Print the header and data for the first few rows found
for row in infobox_rows[:15]:
    th = row.find("th")
    td = row.find("td")
    if th and td:
        print(f"Header (th): {th.get_text(strip=True)}  -->  Data (td): {td.get_text(strip=True)}")

In [ ]:
country = berlin.find("th", string="Country").find_next("td").get_text(strip=True)
country

In [ ]:
latitude = berlin.find("span", class_="latitude").get_text()
longitude = berlin.find("span", class_="longitude").get_text()

population = berlin.find(string="Population").find_next("td").get_text(strip=True)

In [ ]:
print(f"Country: {country}")
print(f"Latitude: {latitude}")
print(f"Longitude: {longitude}")
print(f"Population: {population}")

In [ ]:
#Wikipedia Hamburg
url = "https://en.wikipedia.org/wiki/Hamburg"
headers = {'User-Agent': 'Chrome/134.0.0.0'}

response = requests.get(url, headers=headers)

hamburg = BeautifulSoup(response.content, 'html.parser')

In [ ]:
import re

# 1. Isolate the main infobox on the right side to prevent scraping footer tables
infobox = hamburg.find("table", class_="infobox")

# 2. Extract Country
country = infobox.find("th", string="Country").find_next("td").get_text(strip=True)

# 3. Extract Latitude and Longitude
latitude = infobox.find("span", class_="latitude").get_text()
longitude = infobox.find("span", class_="longitude").get_text()

# 4. Extract Population safely
# This finds the "Population" header, looks forward for "City", then grabs its data cell
pop_header = infobox.find(string=re.compile("Population"))
population = pop_header.find_next(string=re.compile("City")).find_next("td").get_text(strip=True)

# Print your results
print(f"Country: {country}")
print(f"Latitude: {latitude}")
print(f"Longitude: {longitude}")
print(f"Population: {population}")




In [ ]:
#Wikipedia Munich
url = "https://en.wikipedia.org/wiki/Munich"
headers = {'User-Agent': 'Chrome/134.0.0.0'}

response = requests.get(url, headers=headers)

munich = BeautifulSoup(response.content, 'html.parser')

In [ ]:
import re

# 1. Isolate the main infobox on the right side
infobox = munich.find("table", class_="infobox")

# 2. Extract Country
country = infobox.find("th", string="Country").find_next("td").get_text(strip=True)

# 3. Extract Latitude and Longitude
latitude = infobox.find("span", class_="latitude").get_text()
longitude = infobox.find("span", class_="longitude").get_text()

# 4. Extract Population
# Finds the "Population" header segment, looks ahead for "City", and grabs its data cell
pop_header = infobox.find(string=re.compile("Population"))
population = pop_header.find_next(string=re.compile("City")).find_next("td").get_text(strip=True)

# Print your results
print(f"Country: {country}")
print(f"Latitude: {latitude}")
print(f"Longitude: {longitude}")
print(f"Population: {population}")


In [ ]:
#Dataframe cities
import pandas as pd
import re
from sqlalchemy import create_engine

# 1. Provide the data you already gathered
raw_data = [
    {
        "city_name": "Berlin",
        "country": "Germany",
        "latitude": "52°31′N",
        "longitude": "13°24′E",
        "population": "3,700,577"
    },
    {
        "city_name": "Hamburg",
        "country": "Germany",
        "latitude": "53°33′N",
        "longitude": "10°00′E",
        "population": "1,973,896"
    },
    {
        "city_name": "Munich",
        "country": "Germany",
        "latitude": "48°08′N",
        "longitude": "11°34′E",
        "population": "1,505,005"
    }
]

# Helper Function: Converts "53°33′N" to decimal format
def dms_to_decimal(coord_string):
    match = re.match(r"(\d+)°(\d+)′?([NSEW])", coord_string)
    if match:
        degrees = float(match.group(1))
        minutes = float(match.group(2))
        direction = match.group(3)
        
        decimal = degrees + (minutes / 60.0)
        if direction in ['S', 'W']:
            decimal *= -1
        return round(decimal, 3)
    return coord_string

# 2. Process and Clean the existing data inside a loop
cleaned_data = []
for city in raw_data:
    cleaned_data.append({
        "city_name": city["city_name"],
        "country": city["country"],
        "latitude": dms_to_decimal(city["latitude"]),
        "longitude": dms_to_decimal(city["longitude"]),
        "population": int(city["population"].replace(",", ""))  # Converts "1,973,896" to an integer 1973896
    }) 
    # 3. Turn the clean list into a Pandas DataFrame
    df = pd.DataFrame(cleaned_data)
df.insert(0, "city_id", range(1, len(df) + 1))

# Display the finished dataset
df

# 4. Define connection variables
schema = "sql_workshop"
host = "127.0.0.1"
user = "root"
password = "YOUR_MYSQL_PASSWORD" 
port = 3306

# 5. Construct the connection string and engine
connection_string = f'mysql+pymysql://{user}:{password}@{host}:{port}/{schema}'
engine = create_engine(connection_string)

# 6. Push your existing dataframe (df) into the MySQL 'cities' table
df_cities = df[['city_id', 'city_name', 'country', 'latitude', 'longitude']].copy()

df_cities.to_sql(
    "cities",
    con=engine,
    if_exists="replace",
    index=False
)

print("Connected successfully and data uploaded to MySQL!")


In [ ]:
#Dataframe populations
import pandas as pd

df_populations = df[["city_id", "population"]].copy()

df_populations["timestamp_population"] = 2026

df_populations.to_sql(
    "populations",
    con=engine,
    if_exists="replace",
    index=False
)

print("Populations data successfully appended to your local MySQL database!")


In [ ]:
#Weather API
from datetime import datetime

import requests
import pandas as pd
from sqlalchemy import create_engine

# ==========================
# 1. Database Connection
# ==========================

schema = "sql_workshop"
host = "127.0.0.1"
user = "root"
password = "YOUR_MYSQL_PASSWORD"    
port = 3306

connection_string = f"mysql+pymysql://{user}:{password}@{host}:{port}/{schema}"
engine = create_engine(connection_string)

# ==========================
# 2. Read Cities from MySQL
# ==========================

query = """
SELECT city_id, city_name, latitude, longitude
FROM cities;
"""

df_cities = pd.read_sql(query, con=engine)

print(f"Found {len(df_cities)} cities.")

# ==========================
# 3. OpenWeather API
# ==========================

API_KEY = "OPENWEATHERAPIKEY"  

BASE_URL = "https://api.openweathermap.org/data/2.5/forecast"

weather_records = []

# ==========================
# 4. Collect Weather Data
# ==========================

for _, row in df_cities.iterrows():

    city_id = row["city_id"]
    city = row["city_name"]
    lat = row["latitude"]
    lon = row["longitude"]

    print(f"Collecting forecast for {city}...")

    params = {
        "lat": lat,
        "lon": lon,
        "appid": API_KEY,
        "units": "metric"
    }

    try:
        response = requests.get(
            BASE_URL,
            params=params,
            timeout=10
        )

        response.raise_for_status()

        data = response.json()

        for item in data["list"]:

            weather_records.append({
                "city_id": city_id,
                "forecast_time": item["dt_txt"],
                "temperature": item["main"]["temp"],
                "feels_like": item["main"]["feels_like"],
                "humidity": item["main"]["humidity"],
                "pressure": item["main"]["pressure"],
                "weather_main": item["weather"][0]["main"],
                "outlook": item["weather"][0]["description"],
                "wind_speed": item["wind"]["speed"],
                "cloudiness": item["clouds"]["all"],
                "rain_3h_mm": item.get("rain", {}).get("3h", 0),
                "retrieved_at": datetime.now()
            })

    except requests.exceptions.RequestException as e:
        print(f"❌ Error retrieving data for {city}: {e}")

# ==========================
# 5. Create DataFrame
# ==========================

df_weather = pd.DataFrame(weather_records)

print(f"\nCollected {len(df_weather)} forecast records.")

display(df_weather.head())

# ==========================
# 6. Save to MySQL
# ==========================

df_weather.to_sql(
    name="weather",
    con=engine,
    if_exists="replace",
    index=False
)

print("✅ Weather data successfully saved to MySQL!")

In [ ]:
#Flights API
import requests
import pandas as pd
import time
from datetime import datetime, timedelta

# ==========================
# 1. Tomorrow's date
# ==========================

tomorrow = datetime.now() + timedelta(days=1)

date = tomorrow.strftime("%Y-%m-%d")

time_ranges = [
    (f"{date}T00:00", f"{date}T12:00"),
    (f"{date}T12:00", f"{date}T23:59")
]


# ==========================
# 2. Airports
# ==========================

cities = [
    {"city_id": 1, "city_name": "Berlin", "iata": "BER"},
    {"city_id": 2, "city_name": "Hamburg", "iata": "HAM"},
    {"city_id": 3, "city_name": "Munich", "iata": "MUC"},
]


# ==========================
# 3. API headers
# ==========================

headers = {
    "x-rapidapi-key": "X-RAPIDAPI-KEY",
    "x-rapidapi-host": "aerodatabox.p.rapidapi.com",
    "Content-Type": "application/json"
}


querystring = {
    "withLeg": "true",
    "direction": "Arrival",
    "withCancelled": "false",
    "withCodeshared": "true",
    "withCargo": "false",
    "withPrivate": "false",
    "withLocation": "false"
}


# ==========================
# 4. Collect flights
# ==========================

flights = []


for city in cities:

    print(f"Collecting arrivals for {city['city_name']}...")

    for start_time, end_time in time_ranges:

        url = (
            f"https://aerodatabox.p.rapidapi.com/"
            f"flights/airports/iata/{city['iata']}/{start_time}/{end_time}"
        )


        response = requests.get(
            url,
            headers=headers,
            params=querystring
        )


        if response.status_code == 200:

            data = response.json()

            arrivals = data.get("arrivals", [])


            for flight in arrivals:

                flights.append({

                    "city_id": city["city_id"],

                    "airport_iata": city["iata"],

                    "flight_num": flight.get("number"),

                    "airline": flight.get("airline", {}).get("name"),

                    "arrival_time": flight.get("arrival", {})
                        .get("scheduledTime", {})
                        .get("local"),

                    "departure_airport": flight.get("departure", {})
                        .get("airport", {})
                        .get("iata"),

                    "aircraft": flight.get("aircraft", {})
                        .get("model"),

                    "status": flight.get("status")

                })


        else:
            print(
                f"Error {response.status_code} "
                f"for {city['city_name']} "
                f"({start_time} - {end_time})"
            )


        # avoid API rate limit
        time.sleep(2)



# ==========================
# 5. Create DataFrame
# ==========================

df_flights = pd.DataFrame(flights)


print(f"\nCollected {len(df_flights)} flights.")

display(df_flights.head())


# ==========================
# 6. Save to MySQL
# ==========================

df_flights.to_sql(
    "flights",
    con=engine,
    if_exists="replace",
    index=False
)


print("\n✅ Flights successfully saved to MySQL!")

In [ ]:
#Airports API
import requests
import pandas as pd
import time

# ==========================
# 1. Airports to collect
# ==========================

cities = [
    {"city_id": 1, "city_name": "Berlin", "iata": "BER"},
    {"city_id": 2, "city_name": "Hamburg", "iata": "HAM"},
    {"city_id": 3, "city_name": "Munich", "iata": "MUC"},
]

# ==========================
# 2. API Headers
# ==========================

headers = {
    "x-rapidapi-key": "X-RAPIDAPI-KEY",
    "x-rapidapi-host": "aerodatabox.p.rapidapi.com",
    "Content-Type": "application/json"
}

# ==========================
# 3. Collect airport data
# ==========================

airports = []

for city in cities:

    print(f"Collecting airport information for {city['city_name']}...")

    url = f"https://aerodatabox.p.rapidapi.com/airports/iata/{city['iata']}"

    response = requests.get(url, headers=headers)

    if response.status_code == 200:

        airport = response.json()

        airports.append({

            "city_id": city["city_id"],
            "airport_iata": airport.get("iata"),
            "airport_icao": airport.get("icao"),
            "airport_name": airport.get("fullName")

        })

    else:

        print(f"Error {response.status_code} for {city['city_name']}")

    # avoid API rate limit
    time.sleep(2)

# ==========================
# 4. Create DataFrame
# ==========================

df_airports = pd.DataFrame(airports)

print(f"\nCollected {len(df_airports)} airports.")

display(df_airports)

# ==========================
# 5. Save to MySQL
# ==========================

df_airports.to_sql(
    "airports",
    con=engine,
    if_exists="replace",
    index=False
)

print("\n✅ Airports table successfully saved to MySQL!")